# Energy conservation and trajectory accuracy: three horizontal particles

One integration each of BM4Implicit, Gauss–Legendre4, RK4 and DOP853. No runtime benchmarking. This study intentionally uses three methods and DOP853 only, specializing the standard comparison protocol.

The potential is time dependent: physical H is not an invariant. We measure the autonomous energy balance K = H + kappa, with kappa(0)=0 and kappa_dot = -partial_t H. The conservation defect is K(t)-K(0), independently for each particle. RK4 and Gauss advance kappa using their own stages; BM4 reconstructs it from accepted splitting stages; DOP853 integrates the augmented system. No sparse-output quadrature is used. This is a discrete extended-energy diagnostic, not an independently certified continuous-path quadrature.

Keep 200 normalized cycles, 200 fixed steps per cycle and 10 saved intervals per cycle: 40,000 steps, 2,001 saved states including the initial state. DOP853 is adaptive with maximum step 0.005 and the same saved times; its accepted step count can exceed 40,000. No reference agreement audit is claimed.

In [ ]:
from pathlib import Path
import numpy as np
from diagnostics.paths import find_project_root
from diagnostics.dop853_energy_balance_npz import save_energy_balance
from potential import load_gc2d_h5_potential
from studies.dop853_energy_balance import horizontal_configuration, run_energy_balance


## Explicit reproducible configuration

In [ ]:
project_root = find_project_root(Path.cwd())
notebook_directory = project_root / "notebooks/developements/energy/energy_balance_dop853_bm4_gl4_rk4_horizontal_3p_200steps_10saves"
results_path = notebook_directory / "results.npz"
potential_spec = dict(source_path="data/potential/V1/PHI_2.h5", magnetic_field=1.5,
                      characteristic_length=0.06, mode_selection=[0, 1], interpolation_order=3)
initial_spec = dict(x_fractions=[0.25, 0.50, 0.75], y_fraction=0.50)
config = dict(cycles=200, cycle_duration=1.0, steps_per_cycle=200, saves_per_cycle=10,
              rho=0.3, coupling_frequency=float(np.pi/8),
              newton_atol=1e-13, newton_rtol=1e-12, newton_max_iterations=40,
              jacobian_relative_step=float(np.cbrt(np.finfo(float).eps)),
              reference_rtol=1e-10, reference_atol=1e-12)
potential = load_gc2d_h5_potential(project_root / potential_spec['source_path'],
    B=potential_spec['magnetic_field'], characteristic_length=potential_spec['characteristic_length'],
    indx=tuple(potential_spec['mode_selection']), interpolation_order=potential_spec['interpolation_order'])
configuration = horizontal_configuration(potential, **initial_spec)
metadata = dict(schema_version=1, config=config, potential=potential_spec, initial_conditions=initial_spec,
                methods=['DOP853', 'BM4Implicit', 'GaussLegendre4', 'RK4'],
                distance_convention='minimum-image periodic Euclidean',
                energy_diagnostic='K=H+kappa; kappa_dot=-partial_t H; kappa(0)=0',
                multiplier_norm='infinity norm per particle; endpoint and all-step block statistics')
print('Initial state [x1,x2,x3,y1,y2,y3]:', configuration.initial_state)
print('Fixed steps:', config['cycles']*config['steps_per_cycle'])
print('Saved states:', config['cycles']*config['saves_per_cycle']+1)


## Run once and persist

This cell performs the remote calculation when explicitly executed. The NPZ stores aligned states, physical H, kappa, balance defects, periodic distances and BM4 multiplier statistics. The initial multiplier slot is a plotting sentinel, not a nonlinear solve. Each subsequent multiplier block includes 20 accepted steps.

In [ ]:
arrays = run_energy_balance(potential, configuration, config)
save_energy_balance(results_path, arrays, metadata)
print(f"Saved {results_path}")